<a href="https://colab.research.google.com/github/apar-tech/rag-chatbot/blob/main/phase2/langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install Libraries
try:
    !pip install -q \
    langgraph \
    chromadb \
    google-genai \
    groq

    print("Libraries installed successfully!")

except Exception as e:
    print("Installation Error:")
    print(e)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [2]:
#  Import LangGraph
try:
    from typing import TypedDict
    from langgraph.graph import (
        StateGraph,
        START,
        END
    )

    print("LangGraph imports successful!")

except ImportError as e:
    print("Import Error - missing module:")
    print(e)
except Exception as e:
    print("Unexpected import error:")
    print(e)

LangGraph imports successful!


In [3]:
# Define Graph State
try:
    class GraphState(TypedDict):
        question: str
        query_embedding: list
        retrieved_chunks: list
        answer: str

    print("GraphState created successfully!")

except TypeError as e:
    print("Type error in state definition:")
    print(e)
except Exception as e:
    print("State Error:")
    print(e)

GraphState created successfully!


In [4]:
# Initialize Gemini
try:
    from google import genai
    from google.colab import userdata

    gemini_client = genai.Client(
        api_key=userdata.get("geminikey")
    )

    print("Gemini connected successfully!")

except ImportError as e:
    print("Import error - google-genai not installed:")
    print(e)
except KeyError as e:
    print("Key error - geminikey not found in userdata:")
    print(e)
except Exception as e:
    print("Gemini Connection Error:")
    print(e)

Gemini connected successfully!


In [5]:
# Load ChromaDB Data
try:
    import pickle
    import chromadb

    # Load saved data
    with open("chunks.pkl", "rb") as f:
        chunks = pickle.load(f)

    with open("chunk_embeddings.pkl", "rb") as f:
        chunk_embeddings = pickle.load(f)

    print("Files loaded successfully!")

    # Create ChromaDB client
    client = chromadb.PersistentClient(
        path="./digital_chromadb"
    )

    # Create collection
    collection = client.get_or_create_collection(
        name="digital_docs"
    )

    # Add data to collection
    collection.add(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        documents=chunks,
        embeddings=chunk_embeddings
    )

    print("Records:", collection.count())

except FileNotFoundError as e:
    print("File not found error - missing .pkl files:")
    print(e)
except pickle.UnpicklingError as e:
    print("Pickle loading error - corrupted file:")
    print(e)
except chromadb.errors.ChromaError as e:
    print("ChromaDB error:")
    print(e)
except Exception as e:
    print("Error loading data:")
    print(e)

Files loaded successfully!
Records: 111


In [6]:
# Initialize Groq
try:
    from groq import Groq
    from google.colab import userdata

    groq_client = Groq(
        api_key=userdata.get("groqkey")
    )

    print("Groq connected successfully!")

except ImportError as e:
    print("Import error - groq not installed:")
    print(e)
except KeyError as e:
    print("Key error - groqkey not found in userdata:")
    print(e)
except Exception as e:
    print("Groq Connection Error:")
    print(e)

Groq connected successfully!


In [7]:
# Cell 7: Define Embed Question Node
try:
    def embed_question(state):
        try:
            question = state["question"]

            response = gemini_client.models.embed_content(
                model="models/gemini-embedding-001",
                contents=question
            )

            state["query_embedding"] = (
                response.embeddings[0].values
            )

            print("Question embedding generated!")

            return state

        except KeyError as e:
            print(f"Key error in state: {e}")
            raise
        except AttributeError as e:
            print(f"Attribute error - check response structure: {e}")
            raise
        except Exception as e:
            print(f"Error in embed_question: {e}")
            raise

    print("Embed Question Node created!")

except NameError as e:
    print("Name error - required objects not defined:")
    print(e)
except Exception as e:
    print("Node Creation Error:")
    print(e)

Embed Question Node created!


In [8]:
#  Test Embed Question Node
try:
    test_state = {
        "question": "What is machine learning?"
    }

    result = embed_question(test_state)

    print("Embedding Dimension:")
    print(len(result["query_embedding"]))

except NameError as e:
    print("Function not defined:")
    print(e)
except KeyError as e:
    print("Key error in test state:")
    print(e)
except Exception as e:
    print("Test Error:")
    print(e)

Question embedding generated!
Embedding Dimension:
3072


In [9]:
#  Define Retrieve Chunks Node
try:
    def retrieve_chunks(state):
        try:
            results = collection.query(
                query_embeddings=[
                    state["query_embedding"]
                ],
                n_results=5
            )

            state["retrieved_chunks"] = (
                results["documents"][0]
            )

            print("Top 5 chunks retrieved!")

            return state

        except KeyError as e:
            print(f"Key error - missing required field: {e}")
            raise
        except chromadb.errors.ChromaError as e:
            print(f"ChromaDB error: {e}")
            raise
        except Exception as e:
            print(f"Error in retrieve_chunks: {e}")
            raise

    print("Retrieve Chunks Node created!")

except NameError as e:
    print("Name error - collection not defined:")
    print(e)
except Exception as e:
    print("Node Creation Error:")
    print(e)

Retrieve Chunks Node created!


In [10]:
#  Test Retrieve Chunks Node
try:
    state = {
        "question": "What is machine learning?"
    }

    state = embed_question(state)
    state = retrieve_chunks(state)

    print("\nRetrieved Chunks:")
    print(len(state["retrieved_chunks"]))

    # Display retrieved chunks
    for i, chunk in enumerate(
        state["retrieved_chunks"],
        start=1
    ):
        print("=" * 60)
        print(f"Chunk {i}")
        print(chunk[:500])
        print()

except NameError as e:
    print("Function or object not defined:")
    print(e)
except Exception as e:
    print("Test Error:")
    print(e)

Question embedding generated!
Top 5 chunks retrieved!

Retrieved Chunks:
5
Chunk 1
Rule-based models
Rule-based machine learning (RBML) is a branch of machine learning that automatically discovers and learns 'rules' from data. It provides interpretable models, making it useful for decision-making in fields like healthcare, fraud detection, and cybersecurity. Key RBML techniques includes learning classifier systems, association rule learning, artificial immune systems, and other similar models. These methods extract patterns from data and evolve rules over time.

Training model

Chunk 2
Theory
A core objective of a learner is to generalise from its experience. Generalisation in this context is the ability of a learning machine to perform accurately on new, unseen examples/tasks after having experienced a learning data set. The training examples come from some generally unknown probability distribution (considered representative of the space of occurrences) and the learner has to build a

In [11]:
#  Define Generate Answer Node
try:
    def generate_answer(state):
        try:
            context = "\n\n".join(
                state["retrieved_chunks"]
            )

            prompt = f"""
You are a networking assistant.

Answer only using the provided context.

If the answer is not available in the context,
respond with:

I could not find that information in the knowledge base.

Context:
{context}

Question:
{state["question"]}
"""

            response = groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )

            state["answer"] = (
                response.choices[0].message.content
            )

            print("Answer generated successfully!")

            return state

        except KeyError as e:
            print(f"Key error - missing required field: {e}")
            raise
        except AttributeError as e:
            print(f"Attribute error - check response structure: {e}")
            raise
        except Exception as e:
            print(f"Error in generate_answer: {e}")
            raise

    print("Generate Answer Node created!")

except NameError as e:
    print("Name error - groq_client not defined:")
    print(e)
except Exception as e:
    print("Node Creation Error:")
    print(e)

Generate Answer Node created!


In [12]:
# Test Generate Answer Node
try:
    state = {
        "question": "What is machine learning?"
    }

    state = embed_question(state)
    state = retrieve_chunks(state)
    state = generate_answer(state)

    print("\nFinal Answer:\n")
    print(state["answer"])

except NameError as e:
    print("Function or object not defined:")
    print(e)
except Exception as e:
    print("Pipeline Error:")
    print(e)

Question embedding generated!
Top 5 chunks retrieved!
Answer generated successfully!

Final Answer:

Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed.


In [13]:
#  Build Graph - Add Nodes
try:
    graph_builder = StateGraph(GraphState)

    graph_builder.add_node(
        "embed_question",
        embed_question
    )

    graph_builder.add_node(
        "retrieve_chunks",
        retrieve_chunks
    )

    graph_builder.add_node(
        "generate_answer",
        generate_answer
    )

    print("Nodes added successfully!")

except NameError as e:
    print("Name error - GraphState or functions not defined:")
    print(e)
except TypeError as e:
    print("Type error in node addition:")
    print(e)
except Exception as e:
    print("Graph Creation Error:")
    print(e)

Nodes added successfully!


In [14]:
#  Build Graph - Add Edges
try:
    graph_builder.add_edge(
        START,
        "embed_question"
    )

    graph_builder.add_edge(
        "embed_question",
        "retrieve_chunks"
    )

    graph_builder.add_edge(
        "retrieve_chunks",
        "generate_answer"
    )

    graph_builder.add_edge(
        "generate_answer",
        END
    )

    print("Edges connected successfully!")

except AttributeError as e:
    print("Attribute error - graph_builder not properly initialized:")
    print(e)
except Exception as e:
    print("Edge Error:")
    print(e)

Edges connected successfully!


In [15]:
#  Compile Graph
try:
    graph = graph_builder.compile()

    print("LangGraph compiled successfully!")

except AttributeError as e:
    print("Attribute error - graph_builder not ready:")
    print(e)
except Exception as e:
    print("Compilation Error:")
    print(e)

LangGraph compiled successfully!


In [16]:
# Test Graph - Question 1
try:
    result = graph.invoke(
        {
            "question": "What is the purpose of AI?"
        }
    )

    print("\nFinal Answer:\n")
    print(result["answer"])

except KeyError as e:
    print("Key error - missing expected field in result:")
    print(e)
except AttributeError as e:
    print("Attribute error - graph not properly compiled:")
    print(e)
except Exception as e:
    print("Execution Error:")
    print(e)

Question embedding generated!
Top 5 chunks retrieved!
Answer generated successfully!

Final Answer:

Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. The purpose of AI is to develop and study methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.


In [17]:
# Test Graph - Question 2
try:
    result = graph.invoke(
        {
            "question": "What is the difference between machine learning and deep learning?"
        }
    )

    print("\nFinal Answer:\n")
    print(result["answer"])

except KeyError as e:
    print("Key error - missing expected field in result:")
    print(e)
except AttributeError as e:
    print("Attribute error - graph not properly compiled:")
    print(e)
except Exception as e:
    print("Execution Error:")
    print(e)

Question embedding generated!
Top 5 chunks retrieved!
Answer generated successfully!

Final Answer:

Machine learning refers to a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data. Deep learning, on the other hand, is a class of machine learning algorithms that uses a hierarchy of layers to transform input data into a progressively more abstract and composite representation, typically utilizing multi-layered neural networks. In other words, deep learning is a type of machine learning that focuses on using neural networks to perform tasks such as classification, regression, and representation learning.


In [18]:
# Test Graph - Question 3 (Out of Knowledge Base)
try:
    result = graph.invoke(
        {
            "question": "What is the difference between machine language and human language?"
        }
    )

    print("\nFinal Answer:\n")
    print(result["answer"])

except KeyError as e:
    print("Key error - missing expected field in result:")
    print(e)
except AttributeError as e:
    print("Attribute error - graph not properly compiled:")
    print(e)
except Exception as e:
    print("Execution Error:")
    print(e)

Question embedding generated!
Top 5 chunks retrieved!
Answer generated successfully!

Final Answer:

I could not find that information in the knowledge base.


In [19]:
# Interactive Question Function
try:
    def ask_question(question):
        """
        Ask a question using the LangGraph pipeline
        """
        try:
            result = graph.invoke({"question": question})
            return result["answer"]
        except KeyError as e:
            return f"Error: Missing required field - {e}"
        except AttributeError as e:
            return f"Error: Graph not initialized - {e}"
        except Exception as e:
            return f"Error processing question: {e}"

    # Test the function
    test_questions = [
        "What is artificial intelligence?",
        "What are neural networks?",
        "What is the history of machine learning?"
    ]

    for q in test_questions:
        print("=" * 60)
        print(f"Q: {q}")
        print("A:", ask_question(q))
        print()

except NameError as e:
    print("Name error - graph not defined:")
    print(e)
except Exception as e:
    print("Function creation error:")
    print(e)

Q: What is artificial intelligence?
Question embedding generated!
Top 5 chunks retrieved!
Answer generated successfully!
A: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.

Q: What are neural networks?
Question embedding generated!
Top 5 chunks retrieved!
Answer generated successfully!
A: Neural networks, also known as artificial neural networks (ANNs) or connectionist systems, are computing systems inspired by the biological neural networks that constitute animal brains. They are based on a collection of connected units called artificial neurons, 